# Helmet / No Helmet detection - training notebook (YOLO26)

Run the cells **top to bottom**. Before you start:
1. `Runtime` -> `Change runtime type` -> choose **T4 GPU** -> Save.
2. Get a dataset with a *with helmet* and a *without helmet* class (see Step 2).

At the end you download **best.pt** - the trained model - and use it with `detect.py` on your laptop.

## Step 1 - Check the GPU and install libraries

In [ ]:
!nvidia-smi
!pip install -q -U ultralytics roboflow kagglehub

## Step 2 - Get the dataset

Pick ONE source with `DATASET_SOURCE`:

* **`"kaggle_voc"` (default)** - the public *Helmet Detection* dataset on Kaggle (`andrewmvd/helmet-detection`): 764 images, classes *With Helmet* / *Without Helmet*, Pascal VOC labels (converted to YOLO format below).
  If the automatic download fails, download the zip from Kaggle in your browser (free account), upload it to Colab (left sidebar -> Files) as `/content/dataset.zip`, and run the cell again.
* **`"huggingface"`** - the *helmet-detection-dataset* on Hugging Face (`DeBoniz/helmet-detection-dataset`): about 18,000 traffic images, already in YOLO format, classes *Helmet* / *No-Helmet*. Downloads directly, no account needed. Much bigger than the Kaggle one, so Step 3 trains on a random 4,000-image subset to keep Colab fast.
* **`"roboflow"`** - any Roboflow Universe dataset: open its page -> *Download Dataset* -> format **YOLOv8** -> *show download code*, then copy the API key, workspace, project and version into the cell.
* **`"yolo_zip"`** - a zip that is already in YOLO format (images + labels + data.yaml), uploaded as `/content/dataset.zip`.

In [ ]:
DATASET_SOURCE = "kaggle_voc"      # "kaggle_voc" | "huggingface" | "roboflow" | "yolo_zip"
ZIP_PATH = "/content/dataset.zip"  # used if you upload a zip yourself

# ----- only for DATASET_SOURCE = "roboflow" -----
ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY_HERE"
WORKSPACE = "your-workspace"
PROJECT = "your-project"
VERSION = 1

import glob, os, random, shutil, subprocess, zipfile, yaml
import xml.etree.ElementTree as ET
import cv2


def voc_to_yolo(src_dir, out_dir, seed=0, ratios=(0.7, 0.2, 0.1)):
    """Convert Pascal VOC (xml) annotations to a YOLO dataset with train/valid/test folders."""
    imgs = {}
    for p in glob.glob(f"{src_dir}/**/*", recursive=True):
        if p.lower().endswith((".jpg", ".jpeg", ".png")):
            imgs[os.path.splitext(os.path.basename(p))[0]] = p
    items, class_set = [], set()
    for x in sorted(glob.glob(f"{src_dir}/**/*.xml", recursive=True)):
        root = ET.parse(x).getroot()
        stem = os.path.splitext(os.path.basename(x))[0]
        fn = root.findtext("filename") or ""
        img_path = imgs.get(stem) or imgs.get(os.path.splitext(os.path.basename(fn))[0])
        if img_path is None:
            continue
        w = int(float(root.findtext("size/width") or 0))
        h = int(float(root.findtext("size/height") or 0))
        if w <= 0 or h <= 0:
            h, w = cv2.imread(img_path).shape[:2]
        objs = []
        for o in root.findall("object"):
            name = o.findtext("name").strip()
            bb = o.find("bndbox")
            x1, y1, x2, y2 = [float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]
            objs.append((name, x1, y1, x2, y2))
            class_set.add(name)
        items.append((img_path, w, h, objs))
    classes = sorted(class_set)
    random.Random(seed).shuffle(items)
    n = len(items)
    n_tr, n_va = int(ratios[0] * n), int(ratios[1] * n)
    parts = {"train": items[:n_tr], "valid": items[n_tr:n_tr + n_va], "test": items[n_tr + n_va:]}
    shutil.rmtree(out_dir, ignore_errors=True)
    for split, its in parts.items():
        os.makedirs(f"{out_dir}/{split}/images")
        os.makedirs(f"{out_dir}/{split}/labels")
        for img_path, w, h, objs in its:
            stem, ext = os.path.splitext(os.path.basename(img_path))
            shutil.copy(img_path, f"{out_dir}/{split}/images/{stem}{ext}")
            lines = []
            for name, x1, y1, x2, y2 in objs:
                x1, x2 = max(0, min(x1, w)), max(0, min(x2, w))
                y1, y2 = max(0, min(y1, h)), max(0, min(y2, h))
                if x2 <= x1 or y2 <= y1:
                    continue
                lines.append(f"{classes.index(name)} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
            open(f"{out_dir}/{split}/labels/{stem}.txt", "w").write("\n".join(lines) + ("\n" if lines else ""))
    yaml.safe_dump({"names": classes, "nc": len(classes)}, open(f"{out_dir}/data.yaml", "w"))
    print(f"Converted {n} images; classes = {classes}; "
          f"train/valid/test = {len(parts['train'])}/{len(parts['valid'])}/{len(parts['test'])}")


if DATASET_SOURCE == "roboflow":
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    DATASET_DIR = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8").location

elif DATASET_SOURCE in ("yolo_zip", "huggingface"):
    if DATASET_SOURCE == "huggingface":
        url = "https://huggingface.co/datasets/DeBoniz/helmet-detection-dataset/resolve/main/helmet-detection-dataset.zip"
        ZIP_PATH = "/content/hf_dataset.zip"
        if not os.path.exists(ZIP_PATH):
            print("Downloading about 1.5 GB - this can take a few minutes...")
            subprocess.run(["curl", "-L", "-s", "-o", ZIP_PATH, url], check=True)
    print("Unzipping...")
    zipfile.ZipFile(ZIP_PATH).extractall("/content/dataset")

    # find the folder that contains train/images (works for any folder layout)
    shutil.rmtree("/content/dataset/__MACOSX", ignore_errors=True)   # junk folder some zips contain
    roots = [os.path.dirname(os.path.dirname(p))
             for p in glob.glob("/content/dataset/**/train/images", recursive=True)
             if "__MACOSX" not in p]
    roots.sort(key=lambda r: len(glob.glob(f"{r}/train/images/*")), reverse=True)  # real data = most images
    if not roots:
        print(subprocess.run("find /content/dataset -maxdepth 4 | head -40",
                             shell=True, capture_output=True, text=True).stdout)
        raise AssertionError("Could not find train/images inside the zip - send me the list printed above.")
    DATASET_DIR = roots[0]

    # make sure there is a data.yaml with class names
    if not os.path.exists(f"{DATASET_DIR}/data.yaml"):
        other = glob.glob(f"{DATASET_DIR}/*.y*ml")
        if other:
            shutil.copy(other[0], f"{DATASET_DIR}/data.yaml")
        elif DATASET_SOURCE == "huggingface":
            # the zip has no yaml; class order taken from the dataset page: 0 = Helmet, 1 = No-Helmet
            yaml.safe_dump({"names": ["Helmet", "No-Helmet"]}, open(f"{DATASET_DIR}/data.yaml", "w"))
        else:
            raise AssertionError("No data.yaml found - create one containing a 'names' list.")

else:  # kaggle_voc
    raw = None
    if not os.path.exists(ZIP_PATH):
        try:
            import kagglehub
            raw = kagglehub.dataset_download("andrewmvd/helmet-detection")
        except Exception as e:
            print("Automatic download failed:", e)
            print("-> Download the zip from kaggle.com/datasets/andrewmvd/helmet-detection, "
                  "upload it to /content/dataset.zip, and run this cell again.")
    if raw is None and os.path.exists(ZIP_PATH):
        raw = "/content/dataset_raw"
        zipfile.ZipFile(ZIP_PATH).extractall(raw)
    assert raw is not None, "Dataset not found - see the message above."
    DATASET_DIR = "/content/yolo_from_voc"
    voc_to_yolo(raw, DATASET_DIR)

print("Dataset folder:", DATASET_DIR)

## Step 3 - Keep only the two helmet classes and check the data

Some datasets also contain *number plate*, *rider*, etc. This cell keeps just the classes whose name contains "helmet",
copies the data into a clean folder and prints how many examples each class has.
**Look at the numbers** - if one class has far fewer examples than the other, the model will be weaker on it.

**Optional - retraining with your reviewed violations:** upload `review_dataset.zip` (made by `export_review_dataset.py`) to `/content/review_dataset.zip` before running this cell. The examples are added to the training set, so the model learns from the mistakes you corrected.

In [ ]:
import glob, os, random, shutil, yaml
from collections import Counter

src = yaml.safe_load(open(f"{DATASET_DIR}/data.yaml"))
names = src["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
print("Classes in the dataset:", names)

# Keep only helmet classes. If this picks the wrong ones, set KEEP_NAMES by hand, e.g. ["With Helmet", "Without Helmet"]
KEEP_NAMES = [n for n in names if "helmet" in n.lower()]
print("Keeping:", KEEP_NAMES)
assert len(KEEP_NAMES) == 2, "Expected exactly 2 helmet classes - edit KEEP_NAMES by hand."

old2new = {names.index(n): i for i, n in enumerate(KEEP_NAMES)}

MAX_TRAIN_IMAGES = 4000 if DATASET_SOURCE == "huggingface" else None   # None = use everything
CLEAN = "/content/helmet_data"
shutil.rmtree(CLEAN, ignore_errors=True)
splits = [s for s in ["train", "valid", "val", "test"] if os.path.isdir(f"{DATASET_DIR}/{s}/images")]
print("Splits found:", splits)

for split in splits:
    shutil.copytree(f"{DATASET_DIR}/{split}", f"{CLEAN}/{split}")
    if split == "train" and MAX_TRAIN_IMAGES:
        files = sorted(glob.glob(f"{CLEAN}/train/images/*"))
        if len(files) > MAX_TRAIN_IMAGES:
            random.Random(0).shuffle(files)
            for f in files[MAX_TRAIN_IMAGES:]:
                os.remove(f)
                lf = f.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
                if os.path.exists(lf):
                    os.remove(lf)
    for lf in glob.glob(f"{CLEAN}/{split}/labels/*.txt"):
        out = []
        for line in open(lf).read().strip().splitlines():
            parts = line.split()
            if parts and int(float(parts[0])) in old2new:
                out.append(" ".join([str(old2new[int(float(parts[0]))])] + parts[1:]))
        open(lf, "w").write("\n".join(out) + ("\n" if out else ""))

# ----- optional: add reviewed examples (from review_violations.py + export_review_dataset.py) -----
REVIEW_ZIP = "/content/review_dataset.zip"   # upload your zip here to use it
REVIEW_REPEAT = 3                            # copies of each reviewed example (there are few, so repeat them)
if os.path.exists(REVIEW_ZIP):
    import json, zipfile
    rv = "/content/review_data"
    shutil.rmtree(rv, ignore_errors=True)
    zipfile.ZipFile(REVIEW_ZIP).extractall(rv)
    rv_names = json.load(open(f"{rv}/classes.json"))["names"]

    def _is_bad(n):
        n = n.lower()
        return "without" in n or n.startswith("no")

    rv_bad = next(i for i, n in enumerate(rv_names) if _is_bad(n))
    new_bad = next(i for i, n in enumerate(KEEP_NAMES) if _is_bad(n))
    # review class index -> dataset class index (matched by meaning: helmet / no helmet)
    rv_map = {i: (new_bad if i == rv_bad else 1 - new_bad) for i in range(len(rv_names))}

    n_rev = 0
    for img in glob.glob(f"{rv}/images/*.jpg"):
        stem = os.path.splitext(os.path.basename(img))[0]
        lab = f"{rv}/labels/{stem}.txt"
        lines = []
        if os.path.exists(lab):
            for line in open(lab).read().strip().splitlines():
                parts = line.split()
                if parts:
                    lines.append(" ".join([str(rv_map[int(parts[0])])] + parts[1:]))
        for k in range(REVIEW_REPEAT):
            shutil.copy(img, f"{CLEAN}/train/images/rev{k}_{stem}.jpg")
            open(f"{CLEAN}/train/labels/rev{k}_{stem}.txt", "w").write("\n".join(lines) + ("\n" if lines else ""))
        n_rev += 1
    print(f"Added {n_rev} reviewed examples (x{REVIEW_REPEAT}) to the training set")

if "valid" not in splits and "val" not in splits:
    print("No validation split found - using 10% of train for validation")
    os.makedirs(f"{CLEAN}/valid/images")
    os.makedirs(f"{CLEAN}/valid/labels")
    tr = sorted(glob.glob(f"{CLEAN}/train/images/*"))
    random.Random(1).shuffle(tr)
    for f in tr[:max(1, len(tr) // 10)]:
        lf = f.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        shutil.move(f, f"{CLEAN}/valid/images/" + os.path.basename(f))
        if os.path.exists(lf):
            shutil.move(lf, f"{CLEAN}/valid/labels/" + os.path.basename(lf))
    splits.append("valid")

val_split = "valid" if "valid" in splits else "val"
cfg = {"path": CLEAN, "train": "train/images", "val": f"{val_split}/images",
       "names": {i: n for i, n in enumerate(KEEP_NAMES)}}
if "test" in splits:
    cfg["test"] = "test/images"
yaml.safe_dump(cfg, open(f"{CLEAN}/data.yaml", "w"))
DATA_YAML = f"{CLEAN}/data.yaml"

for split in splits:
    cnt, n_img = Counter(), len(glob.glob(f"{CLEAN}/{split}/images/*"))
    for lf in glob.glob(f"{CLEAN}/{split}/labels/*.txt"):
        for line in open(lf):
            if line.strip():
                cnt[KEEP_NAMES[int(line.split()[0])]] += 1
    print(f"{split:6s} images: {n_img:5d}  boxes per class: {dict(cnt)}")

## Step 4 - Look at a few training images (sanity check)
Make sure the boxes sit on helmets / bare heads. Wrong labels = a wrong model.

In [ ]:
import random, cv2
import matplotlib.pyplot as plt

imgs = glob.glob(f"{CLEAN}/train/images/*")
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, p in zip(axes.flat, random.sample(imgs, 6)):
    im = cv2.imread(p); h, w = im.shape[:2]
    lf = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if os.path.exists(lf):
        for line in open(lf):
            c, x, y, bw, bh = line.split()[:5]
            x, y, bw, bh = float(x) * w, float(y) * h, float(bw) * w, float(bh) * h
            col = (0, 200, 0) if int(c) == 0 else (0, 0, 255)
            cv2.rectangle(im, (int(x - bw/2), int(y - bh/2)), (int(x + bw/2), int(y + bh/2)), col, 3)
    ax.imshow(im[..., ::-1]); ax.axis("off")
plt.suptitle(f"green = {KEEP_NAMES[0]}, red = {KEEP_NAMES[1]}")
plt.show()

## Step 5 - Train
`yolo26n` is the smallest and fastest model. If accuracy is not good enough, change it to `yolo26s`.
If YOLO26 gives any problem, use `yolo11n.pt` instead - everything else stays the same.
Training stops early if it stops improving (`patience`).

In [ ]:
from ultralytics import YOLO

START_FROM = "yolo26n.pt"      # alternatives: "yolo26s.pt", "yolo11n.pt"
EPOCHS = 45
# To IMPROVE your earlier model with reviewed examples: upload best.pt to Colab, then set
#   START_FROM = "/content/best.pt"   and   EPOCHS = 20

model = YOLO(START_FROM)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    patience=15,
    seed=0,
    project="helmet_runs",
    name="yolo26n",
)
SAVE_DIR = str(model.trainer.save_dir)
BEST = f"{SAVE_DIR}/weights/best.pt"
print("Best model:", BEST)

## Step 6 - How good is it? (use these numbers in your report)

In [ ]:
best = YOLO(BEST)
split = "test" if "test" in splits else "val"
m = best.val(data=DATA_YAML, split=split)
print(f"\nEvaluated on the {split} set")
print(f"Precision : {m.box.mp:.3f}")
print(f"Recall    : {m.box.mr:.3f}")
print(f"mAP@0.5   : {m.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {m.box.map:.3f}")
for i, n in enumerate(KEEP_NAMES):
    print(f"  {n}: AP50 = {m.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display
for f in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = f"{SAVE_DIR}/{f}"
    if os.path.exists(p):
        print(f); display(Image(p, width=800))

## Step 7 - Try it on test photos
Look at the mistakes too - they tell you what to improve (more data, different angles, night photos...).

In [ ]:
from PIL import Image as PILImage
from IPython.display import display

test_dir = f"{CLEAN}/test/images" if "test" in splits else f"{CLEAN}/{val_split}/images"
files = sorted(glob.glob(f"{test_dir}/*"))[:8]
for r in best.predict(files, conf=0.4, verbose=False):
    display(PILImage.fromarray(r.plot()[..., ::-1]).resize((480, 360)))

## Step 8 - Download the trained model
Put the downloaded **best.pt** in the same folder as `detect.py` on your laptop.

In [ ]:
from google.colab import files
files.download(BEST)
# Optional: keep a copy in Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy(BEST, '/content/drive/MyDrive/helmet_best.pt')